In [1]:
# %pip install pandas holidays numpy requests fastparquet pyarrow prophet

In [2]:
import os
import pandas as pd
import numpy as np
import requests
import holidays
from datetime import datetime, timedelta
import io

EIA_API_KEY = os.getenv("EIA_API_KEY")


### Data Extraction

In [3]:
# Mapping cities to their Latitude, Longitude, and EIA Balancing Authority (Grid Region)
# Electricity is tracked by grid region, not city limits. 
CITIES = {
    "Chicago": {
        "lat": 41.8781,
        "lon": -87.6298,
        "eia_region": "PJM",
        "STATE_NAME": "Illinois",
        "COUNTY_NAME": "Cook",
    }
    , "New York": {
        "lat": 40.7128,
        "lon": -74.0060,
        "eia_region": "NYIS",
        "STATE_NAME": "New York",
        "COUNTY_NAME": "New York",
    }
    , "Seattle": {
        "lat": 47.6062,
        "lon": -122.3321,
        "eia_region": "BPAT",
        "STATE_NAME": "Washington",
        "COUNTY_NAME": "King",
    }
    , "Miami": {
        "lat": 25.7617,
        "lon": -80.1918,
        "eia_region": "FPL",
        "STATE_NAME": "Florida",
        "COUNTY_NAME": "Miami-Dade",
    }
    , "Dallas": {
        "lat": 32.7767,
        "lon": -96.7970,
        "eia_region": "ERCO",
        "STATE_NAME": "Texas",
        "COUNTY_NAME": "Dallas",
    },
}

# Date ranges: 3 years ago to 14 days in the future
today = datetime.today()
start_date = (today - timedelta(days=3*365)).strftime('%Y-%m-%d')
forecast_end = (today + timedelta(days=14)).strftime('%Y-%m-%d')
today_str = today.strftime('%Y-%m-%d')

# --- 1. FETCH WEATHER DATA (Historical + Forecast) ---
def get_weather_data(city, lat, lon):
    print(f"Fetching weather for {city}...")
    
    # Open-Meteo Historical Archive API
    hist_url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={start_date}&end_date={today_str}&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,relative_humidity_2m_mean,wind_speed_10m_max,shortwave_radiation_sum&timezone=auto"
    
    # Open-Meteo Forecast API
    forecast_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&start_date={today_str}&end_date={forecast_end}&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,relative_humidity_2m_mean,wind_speed_10m_max,shortwave_radiation_sum&timezone=auto"

    hist_data = requests.get(hist_url).json()
    forecast_data = requests.get(forecast_url).json()
    
    # Convert to DataFrames
    df_hist = pd.DataFrame(hist_data['daily'])
    df_cast = pd.DataFrame(forecast_data['daily'])

    
    # Combine and drop the overlapping 'today' row
    df_weather = pd.concat([df_hist, df_cast]).drop_duplicates(subset=['time'], keep='last')
    
    # Rename columns for clarity
    df_weather.rename(columns={
        'time': 'Date',
        'temperature_2m_max': 'Temp_Max_C',
        'temperature_2m_min': 'Temp_Min_C',
        'temperature_2m_mean': 'Temp_Mean_C',
        'relative_humidity_2m_mean': 'Humidity_Pct',
        'wind_speed_10m_max': 'Wind_Speed_kmh',
        'shortwave_radiation_sum': 'Solar_Irradiance_MJm2'
    }, inplace=True)
    
    # Calculate Heating Degree Days (HDD) and Cooling Degree Days (CDD)
    # Baseline is typically 18.3°C (65°F)
    baseline_c = 18.3
    df_weather['HDD'] = df_weather['Temp_Mean_C'].apply(lambda x: max(0, baseline_c - x))
    df_weather['CDD'] = df_weather['Temp_Mean_C'].apply(lambda x: max(0, x - baseline_c))
    
    df_weather['City'] = city
    return df_weather

# --- 2. FETCH EIA ELECTRICITY DATA ---
def get_eia_data(eia_region):
    print(f"Fetching EIA data for region {eia_region}...")
    
    # Actual Demand
    url = f"https://api.eia.gov/v2/electricity/rto/daily-region-data/data/?api_key={EIA_API_KEY}&frequency=daily&data[0]=value&facets[respondent][]={eia_region}&facets[type][]=D&facets[timezone][]=Eastern&start={start_date}&end={today_str}&sort[0][column]=period&sort[0][direction]=desc&length=3000"
    response = requests.get(url)
    
    # Enhanced error logging to print the exact rejection reason from the API
    if response.status_code != 200:
        print(f"Warning: Failed to fetch EIA data for {eia_region}.")
        print(f"Status Code: {response.status_code}")
        print(f"API Message: {response.text}")
        return pd.DataFrame(columns=['Date', 'MWh_Consumed'])
        
    data = response.json()
    
    if 'response' in data and 'data' in data['response']:
        df_eia = pd.DataFrame(data['response']['data'])
        if not df_eia.empty:
            df_eia = df_eia[df_eia['timezone'] == 'Eastern']
            df_eia = df_eia[['period', 'value']]
            df_eia.rename(columns={'period': 'Date', 'value': 'MWh_Consumed'}, inplace=True)
            return df_eia
            
    return pd.DataFrame(columns=['Date', 'MWh_Consumed'])

# --- 3. MERGE EVERYTHING TOGETHER ---
df_cities_loop = []

for city, info in CITIES.items():
    # Fetch weather
    df_weather = get_weather_data(city, info["lat"], info["lon"])
    df_weather["STATE_NAME"] = info["STATE_NAME"]
    df_weather["COUNTY_NAME"] = info["COUNTY_NAME"]

    # Fetch EIA energy data
    df_eia = get_eia_data(info["eia_region"])

    # Merge weather and EIA
    df_merged = pd.merge(df_weather, df_eia, on="Date", how="left")
    df_cities_loop.append(df_merged)

# Final single DataFrame
df_final = pd.concat(df_cities_loop, ignore_index=True)

# Reset index 
df_final = df_final.reset_index(drop=True)

# Convert Date column to actual datetime objects
df_final['Date'] = pd.to_datetime(df_final['Date'])

# Convert MWh_Consumed to numeric, coercing errors to NaN
df_final['MWh_Consumed'] = pd.to_numeric(df_final['MWh_Consumed'], errors='coerce')

print("\nDataframe constructed successfully!")
# print(df_final.head())\



Fetching weather for Chicago...
Fetching EIA data for region PJM...
Fetching weather for New York...
Fetching EIA data for region NYIS...
Fetching weather for Seattle...
Fetching EIA data for region BPAT...
Fetching weather for Miami...
Fetching EIA data for region FPL...
Fetching weather for Dallas...
Fetching EIA data for region ERCO...

Dataframe constructed successfully!


In [4]:
# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#     # display(df_final.head(1000))
#     display(df_final[df_final['City'] == 'Chicago'])

### Data Cleanup

In [5]:
def fill_mwh_nulls(df):
    """
    Fills null values in MWh_Consumed using the average of the prior and following dates.
    Only applies to historical data (before today's date).
    """
    # Define 'today' to match the start of the current day
    today = pd.to_datetime(datetime.today().strftime('%Y-%m-%d'))
    
    # 1. Sort the data to ensure chronological order for interpolation
    df = df.sort_values(by=['City', 'Date'])
    
    # 2. Group by city and linearly interpolate
    interpolated_mwh = df.groupby('City')['MWh_Consumed'].transform(lambda x: x.interpolate(method='linear'))
    
    # 3. Create a mask for historical dates
    historical_mask = df['Date'] < today
    
    # 4. Fill nulls ONLY where the date is before today
    df.loc[historical_mask, 'MWh_Consumed'] = df.loc[historical_mask, 'MWh_Consumed'].fillna(interpolated_mwh)
    
    return df

def fill_numerical_nulls(df):
    """
    Fills null values for all numerical columns (except MWh_Consumed) 
    using linear interpolation, regardless of the date.
    """
    # 1. Sort to ensure chronological order for interpolation
    df = df.sort_values(by=['City', 'Date'])
    
    # 2. Identify all numerical columns (floats and ints)
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    
    # 3. Exclude 'MWh_Consumed' so we don't fill the future forecast dates
    if 'MWh_Consumed' in numeric_cols:
        numeric_cols.remove('MWh_Consumed')
        
    # 4. Group by city and interpolate for each remaining numerical column
    for col in numeric_cols:
        df[col] = df.groupby('City')[col].transform(
            # Using limit_direction='both' ensures it fills leading/trailing nulls 
            # if they occur at the very beginning or end of a city's dataset
            lambda x: x.interpolate(method='linear', limit_direction='both')
        )
        
    return df

# Apply the functions to your final DataFrame
df_final = fill_mwh_nulls(df_final)
df_final = fill_numerical_nulls(df_final)

### Additional Features

#### Temporal features

In [6]:
# Ensure'Date' column is a proper datetime object
df_final['Date'] = pd.to_datetime(df_final['Date'])

# 1. Day of the Week (Monday=0, Sunday=6)
df_final['DayOfWeek'] = df_final['Date'].dt.dayofweek

# 2. Month (1 to 12)
df_final['Month'] = df_final['Date'].dt.month

# 3. Year (e.g., 2023, 2024)
df_final['Year'] = df_final['Date'].dt.year

# 4. Cyclical encoding for Month (12 months in a year)
df_final['Month_Sin'] = np.sin(2 * np.pi * df_final['Month'] / 12)
df_final['Month_Cos'] = np.cos(2 * np.pi * df_final['Month'] / 12)

# 5. Cyclical encoding for Day of the Week (7 days in a week)
df_final['DayOfWeek_Sin'] = np.sin(2 * np.pi * df_final['DayOfWeek'] / 7)
df_final['DayOfWeek_Cos'] = np.cos(2 * np.pi * df_final['DayOfWeek'] / 7)



#### Holiday one-hot    

In [7]:
# 1. Initialize US holidays for your dataset's specific years
unique_years = df_final['Year'].unique().tolist()
unique_years.append(max(unique_years) + 1) ## In cae we are forecasting during New Year's Day, we want to capture that holiday as well.
us_holidays = holidays.US(years=unique_years)

# 2. Map each date to its holiday name (or 'Not_Holiday' for normal days)
df_final['Holiday_Name'] = df_final['Date'].apply(
    lambda x: us_holidays.get(x, 'Not_Holiday')
)

# 3. Clean up the holiday names so they make clean column headers (e.g., remove spaces/apostrophes)
cleaned_holiday_names = (
    df_final['Holiday_Name']
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", "_", regex=True)
)

df_final['Holiday_Clean'] = "Holiday_" + cleaned_holiday_names

# 4. One-hot encode the holidays
# prefix='' prevents duplicate prefixes since we manually added 'Holiday_' above
dummies = pd.get_dummies(df_final['Holiday_Clean'], dtype=int)

# 5. Join the new one-hot columns back to your main dataframe and drop helpers
df_final = pd.concat([df_final, dummies], axis=1)
df_final.drop(columns=['Holiday_Name', 'Holiday_Clean'], inplace=True)
# df_final

In [8]:
# 1. Fetch existing Parquet from GitHub Releases
print("Fetching latest release data from GitHub...")
repo_url = "https://api.github.com/repos/JesusASantillanMinila/data_centers_time_series/releases/latest"
response = requests.get(repo_url)

if response.status_code == 200:
    release_data = response.json()
    # Find the parquet file in the release assets
    parquet_asset = next((asset for asset in release_data.get('assets', []) if asset['name'].endswith('.parquet')), None)
    
    if parquet_asset:
        parquet_url = parquet_asset['browser_download_url']
        print(f"Downloading existing parquet from: {parquet_url}")
        df_data_centers = pd.read_parquet(parquet_url, columns = [
                                                                    "snapshot_date"
                                                                    , "STATE_NAME"
                                                                    , "COUNTY_NAME"
                                                                    ,"total_data_center_count"])
        df_data_centers['snapshot_date'] = pd.to_datetime(df_data_centers['snapshot_date'])

# Join data to main dataframe
# 1. Ensure snapshot_date in df_data_centers is datetime
df_data_centers["snapshot_date"] = pd.to_datetime(
    df_data_centers["snapshot_date"]
)

# 2. Left join df_final with df_data_centers
df_final = pd.merge(
    df_final,
    df_data_centers,
    left_on=["Date", "STATE_NAME", "COUNTY_NAME"],
    right_on=["snapshot_date", "STATE_NAME", "COUNTY_NAME"],
    how="left",
)

# Drop redundant snapshot_date column if present
if "snapshot_date" in df_final.columns:
    df_final.drop(columns=["snapshot_date"], inplace=True)

# 3. Identify data center metric columns
data_center_cols = [
    "total_data_center_count"
]

# Ensure chronological order per city before filling
df_final = df_final.sort_values(by=["City", "Date"]).reset_index(drop=True)

# 4. Forward fill across dates per City to carry forward last known snapshot values
df_final[data_center_cols] = df_final.groupby("City")[
    data_center_cols
].transform(lambda group: group.ffill().fillna(0))

Fetching latest release data from GitHub...


In [9]:
# df_data_centers

In [10]:
# with pd.option_context('display.max_rows', 6000, 'display.max_columns', 50):
#     display(df_final)